In [3]:
import pandas as pd

df = pd.read_excel('/Users/gabriellabollicimoreno/Desktop/DEPRESSION VLOGS/Turkish_Dataset.xlsx', sheet_name='Sheet1')
df = df.dropna(how='all')
df = df.rename(columns={'YouTube Transcript': 'Transcript'})
df = df[["id", "Transcript"]]
df = df[df['Transcript'] != 'Not available']

df.head(10)

,id,Transcript
0,0,mi Evet günaydınlar herkese Merhabalar başlıkt...
1,1,Evet arkadaşlar selam realistik kanalıma Hoşge...
2,2,Evet arkadaşlar Merhabalar günaydınlar depresy...
3,3,Herkese selamlar ayrılıkla mücadele edememe de...
4,4,Evet Merhabalar hoş geldik Hoş bulduk Nasılsın...
5,5,Evet Merhabalar hoş geldik Hoş bulduk Evet bug...
6,6,Merhabalar tekrardan Hoş geldiniz mi hoş mu bu...
7,7,Günaydınlar tünaydınlar artık kaçtı açtıysanız...
8,8,Evet arkadaşlar Herkese selamlar saygılar Hoş ...
9,9,Evet Günaydınlar günaydınlar herkese Merhabala...


In [4]:
pip install https://huggingface.co/turkish-nlp-suite/tr_core_news_md/resolve/main/tr_core_news_md-1.0-py3-none-any.whl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.0/156.0 MB 9.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.6/152.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.0/57.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 919.6/919.6 kB 53.9 MB/s eta 0:00:00
  Attempting uninstall: wasabi
    Found existing installation: wasabi 1.1.3
    Uninstalling wasabi-1.1.3:
      Successfully uninstalled wasabi-1.1.3
  Attempting uninstall: typer
    Found existing installation: typer 0.13.0
    Uninstalling typer-0.13.0:
      Successfully uninstalled typer-0.13.0
  Attempting uninstall: smart-open
    Found existing installation: smart-open 7.0.5
    Uninstalling smart-open-7.0.5:
      Successfully uni

In [5]:
!pip install langdetect
!pip install spacy
!pip install stanza

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 18.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993222 sha256=4ba3d1652308fa2415464558747f47f7c2c10b098e96019d46cc2b034b9266a5
  Stored in directory: /root/.cache/pip/wheels/95/03/7d/59ea870c70ce4e5a370638b5462a7711ab78fba2f655d05106
Successfully built langdetect
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 586.9/586.9 kB 41.2 MB/s eta 0:00:00


In [6]:
import pandas as pd
from langdetect import detect
import re
import spacy
from spacy.matcher import PhraseMatcher
from spacy.tokens import Span

transcript_column = [col for col in df.columns if col.startswith('Transcript')][0]
detected_language = "turkish"

nlp = spacy.load("tr_core_news_md")

def remove_text_in_brackets(text):
    return re.sub(r'\[.*?\]|\(.*?\)', '', text)

df['clean_transcript'] = df[transcript_column].apply(remove_text_in_brackets)

def process_doc(doc):
    tokens = [token.text for token in doc]
    pos_tags = [(token.text, token.pos_) for token in doc]

    if 'lemmatizer' in nlp.pipe_names:
        lemma = [(token.text, token.lemma_) for token in doc]
    else:
        lemma = [(token.text, None) for token in doc]

    entity = [(ent.text, ent.start_char, ent.end_char, ent.label_) for ent in doc.ents]

    return tokens, pos_tags, lemma, entity

mental_health_words_tk = """Üzgün, Mutlu, Umutsuz, Umutlu, Boş, Tatmin olmuş, Tatminsiz, Hüzünlü, Sevinçli, Umutsuzluk, Umut, Düşük, Yüksek, Uyuşuk, Uyanık, Melankolik,
Mesut, Yenik, Başarılı, Hüzünlü, Hüzünlü, Karamsar, İyimser, Kasvetli, Parlak, Cesaretsiz, İlham almak, Neşesiz, Neşeli, Duyarsız, Duyarlı, Kaygısız, Özenli, Motivasyonsuz,
Motive, Bağlantısız, Bağlı, Sıkılmış, İlgi duyan, Soyutlanmış, İlgili, Gerçekleşmemiş, Gerçekleşmiş, Hayatsız, Canlı, Bağlantısız, Bağlantılı, Boş, İfade edici, Duygusuz,
Duygusal, Suçlu, Masum, Değersiz, Değerli, Utanmış, Gururlu, Başarısız, Başarılı, Yetersiz, Yetenekli, Kendini suçlama, Kendini kabul etme, Pişmanlık duyan, Memnun, İşe yaramaz,
Faydalı, Kusurlu, Mükemmel, Mahcup, Kendine güvenen, Değersiz, Değerli, Hak etmeyen, Hak eden, Yorgun, Enerjik, Tükenmiş, Yenilenmiş, Bitik, Yenilenmiş, Yorgun, Aşırı kilolu,
Aktif, Enerjik, Canlı, Uyuşuk, Uyanık, Baş ağrısı, Rahatlama, Bulantı, Kas ağrısı, Esneklik, Karın ağrısı, Rahatlık, Ağrı, Zayıf, Güçlü, Baş dönmesi, Denge, Ağrıyan, Sıkı,
Gevşek, Ağrı, Sağlıklı, Uykusuzluk, Uykusuz, Uykulu, Uyumakta, Aşırı uyuma, Kesintiye uğramış, Kesintisiz, Kâbus, Rüya, Kırık, Uykusuz, İyi dinlenmiş, İştahsız, İştah,
İhtiyaçlar, Tatmin, Aşırı yeme, Zayıf, Sağlıksız, Sağlıklı, Dengeli, Yetersiz beslenmiş, İzole olmuş, Yalnız, Birlikte, İçine kapanık, Reddedilmiş, Kabul edilmiş, Sevilmeyen,
Sevilen, Yanlış anlaşılan, Anlaşılan, İhmal edilen, Özen gösterilen, Kaçınılan, Kucaklanan, Anlamsız, Değerli, Önemli, Kasvetli, Anlamsız, Anlamlı, Boşuna, Verimli, Tam, Başarı,
Güven, Karamsar, Cesaretlendirilmiş, Mahkum, Dayanıklı, Amaçsız, Hedefli, Güvenli, Güvensiz, Yetersiz, Yeterli"""


mental_health_words_tk = [word.strip().lower() for word in mental_health_words_tk.split(',')]

matcher = PhraseMatcher(nlp.vocab)
patterns = [nlp.make_doc(word) for word in mental_health_words_tk]
matcher.add("MENTAL_HEALTH", patterns)

def detect_and_highlight_mental_health_entities(doc):
    matches = matcher(doc)
    spans = [Span(doc, start, end, label="MENTAL_HEALTH") for match_id, start, end in matches]
    spans = spacy.util.filter_spans(spans)
    existing_ents = list(doc.ents)
    non_conflicting_spans = [span for span in spans if not any(ent.start < span.end and span.start < ent.end for ent in existing_ents)]
    doc.ents = existing_ents + non_conflicting_spans

    highlighted_text = doc.text
    offset = 0
    for ent in doc.ents:
        if ent.label_ == "MENTAL_HEALTH":
            start = ent.start_char + offset
            end = ent.end_char + offset
            highlighted_text = highlighted_text[:start] + f"<{ent.label_}>" + highlighted_text[start:end] + f"</{ent.label_}>" + highlighted_text[end:]
            offset += len(f"<{ent.label_}></{ent.label_}>")

    mental_health_entities = [(ent.text, ent.label_) for ent in doc.ents if ent.label_ == "MENTAL_HEALTH"]
    return highlighted_text, mental_health_entities

def process_text(text):
    doc = nlp(text)
    tokens, pos_tags, lemma, entity = process_doc(doc)
    highlighted_text, mental_health_entities = detect_and_highlight_mental_health_entities(doc)
    return tokens, pos_tags, lemma, entity, highlighted_text, mental_health_entities

df[['tokens', 'pos_tag', 'lemma', 'entity', 'highlighted_transcript', 'mental_health_entities']] = df['clean_transcript'].apply(lambda text: pd.Series(process_text(text)))
display(df[[transcript_column, 'clean_transcript', 'tokens', 'pos_tag', 'lemma', 'entity', 'highlighted_transcript', 'mental_health_entities']])


,Transcript,clean_transcript,tokens,pos_tag,lemma,entity,highlighted_transcript,mental_health_entities
0,mi Evet günaydınlar herkese Merhabalar başlıkt...,mi Evet günaydınlar herkese Merhabalar başlıkt...,"[mi, Evet, günaydınlar, herkese, Merhabalar, b...","[(mi, AUX), (Evet, NOUN), (günaydınlar, NOUN),...","[(mi, None), (Evet, None), (günaydınlar, None)...","[(Ben, 191, 194, PERSON), (Ben, 216, 219, PERS...",mi Evet günaydınlar herkese Merhabalar başlıkt...,"[(üzgün, MENTAL_HEALTH), (önemli, MENTAL_HEALT..."
1,Evet arkadaşlar selam realistik kanalıma Hoşge...,Evet arkadaşlar selam realistik kanalıma Hoşge...,"[Evet, arkadaşlar, selam, realistik, kanalıma,...","[(Evet, NOUN), (arkadaşlar, NOUN), (selam, NOU...","[(Evet, None), (arkadaşlar, None), (selam, Non...","[(Hoşgeldiniz, 41, 52, PERSON), (ikinci, 66, 7...",Evet arkadaşlar selam realistik kanalıma Hoşge...,"[(bitik, MENTAL_HEALTH), (boş, MENTAL_HEALTH),..."
2,Evet arkadaşlar Merhabalar günaydınlar depresy...,Evet arkadaşlar Merhabalar günaydınlar depresy...,"[Evet, arkadaşlar, Merhabalar, günaydınlar, de...","[(Evet, NOUN), (arkadaşlar, NOUN), (Merhabalar...","[(Evet, None), (arkadaşlar, None), (Merhabalar...","[(Merhabalar, 16, 26, PERSON), (Evet, 209, 213...",Evet arkadaşlar Merhabalar günaydınlar depresy...,"[(önemli, MENTAL_HEALTH), (sağlıklı, MENTAL_HE..."
3,Herkese selamlar ayrılıkla mücadele edememe de...,Herkese selamlar ayrılıkla mücadele edememe de...,"[Herkese, selamlar, ayrılıkla, mücadele, edeme...","[(Herkese, NOUN), (selamlar, NOUN), (ayrılıkla...","[(Herkese, None), (selamlar, None), (ayrılıkla...","[(Herkese, 0, 7, PERSON), (Hoş geldiniz, 82, 9...",Herkese selamlar ayrılıkla mücadele edememe de...,"[(tam, MENTAL_HEALTH), (önemli, MENTAL_HEALTH)..."
4,Evet Merhabalar hoş geldik Hoş bulduk Nasılsın...,Evet Merhabalar hoş geldik Hoş bulduk Nasılsın...,"[Evet, Merhabalar, hoş, geldik, Hoş, bulduk, N...","[(Evet, NOUN), (Merhabalar, NOUN), (hoş, ADV),...","[(Evet, None), (Merhabalar, None), (hoş, None)...","[(Evet Merhabalar, 0, 15, WORK_OF_ART), (Nasıl...",Evet Merhabalar hoş geldik Hoş bulduk Nasılsın...,"[(zayıf, MENTAL_HEALTH), (ağrı, MENTAL_HEALTH)..."
...,...,...,...,...,...,...,...,...
106,Selam arkadaşlar ben Esra sizinle daha önceden...,Selam arkadaşlar ben Esra sizinle daha önceden...,"[Selam, arkadaşlar, ben, Esra, sizinle, daha, ...","[(Selam, NOUN), (arkadaşlar, NOUN), (ben, PRON...","[(Selam, None), (arkadaşlar, None), (ben, None...","[(Esra, 21, 25, PERSON), (YouTube'dan, 47, 58,...",Selam arkadaşlar ben Esra sizinle daha önceden...,[]
107,Merhaba kanalıma hoş geldiniz Ben Esra bugün s...,Merhaba kanalıma hoş geldiniz Ben Esra bugün s...,"[Merhaba, kanalıma, hoş, geldiniz, Ben, Esra, ...","[(Merhaba, NOUN), (kanalıma, NOUN), (hoş, ADJ)...","[(Merhaba, None), (kanalıma, None), (hoş, None...","[(Ben Esra, 30, 38, PERSON), (8 yıldır, 175, 1...",Merhaba kanalıma hoş geldiniz Ben Esra bugün s...,"[(yüksek, MENTAL_HEALTH), (mutlu, MENTAL_HEALT..."
108,bu kanalıma hepiniz hoş geldiniz ancak Herkese...,bu kanalıma hepiniz hoş geldiniz ancak Herkese...,"[bu, kanalıma, hepiniz, hoş, geldiniz, ancak, ...","[(bu, DET), (kanalıma, NOUN), (hepiniz, NOUN),...","[(bu, None), (kanalıma, None), (hepiniz, None)...","[(Herkese, 39, 46, PERSON), (Bundan Yaklaşık, ...",bu kanalıma hepiniz hoş geldiniz ancak Herkese...,"[(tam, MENTAL_HEALTH), (tam, MENTAL_HEALTH)]"
109,e herkese selam arkadaşlar ben Esra daha önces...,e herkese selam arkadaşlar ben Esra daha önces...,"[e, herkese, selam, arkadaşlar, ben, Esra, dah...","[(e, NOUN), (herkese, NOUN), (selam, NOUN), (a...","[(e, None), (herkese, None), (selam, None), (a...","[(Esra, 31, 35, PERSON), (birkaç, 54, 60, CARD...",e herkese selam arkadaşlar ben Esra daha önces...,"[(düşük, MENTAL_HEALTH), (faydalı, MENTAL_HEAL..."


In [7]:
def Pronoun_analysis(doc_id, text, language):
    """Analyze the use of first-person singular and plural pronouns in the text and return counts for each."""
    fps_counts = []
    fpp_counts = []
    sps_counts = []
    spp_counts = []
    tps_counts = []
    tpp_counts = []
    FPS = 0
    FPP = 0
    SPS = 0
    SPP = 0
    TPS = 0
    TPP = 0

    if language == 'turkish':
        fps_list = ['Ben', 'Beni', 'Bana', 'Benim', 'Bende', 'Benden', 'Benle', 'Benimle', 'Benle', 'Bensiz', 'Benimki', 'Bencileyin', 'Benimki', 'Bence']
        fpp_list = ['Biz', 'Bizi', 'Bize', 'Bizim', 'Bizde', 'Bizden', 'Bizle', 'Bizimle', 'Bizsiz', 'Bizimki', 'Bizce']
        sps_list = ['Sen','Seni','Sana','Senin','Sende','Senden', 'Senle','Seninle','Sensiz','Seninki','Sencileyin', 'Seninki','Sence']
        spp_list = ['Siz', 'Sizi', 'Size', 'Sizin', 'Sizde', 'Sizden','Sizle','Sizimle','Sizsiz','Sizinki','Sizce']
        tps_list = ['O','Onu','Ona','Onun','Onda','Ondan','Onla','Onunla','Onsuz','Onunki']
        tpp_list = ['Onlar', 'Onları', 'Onlara', 'Onların','Onlarda','Onlardan','Onlarla','Onlarsız','Onlarınki']

        word_counts_fps = {pronoun: text.split().count(pronoun) for pronoun in fps_list}
        word_counts_fpp = {pronoun: text.split().count(pronoun) for pronoun in fpp_list}
        word_counts_sps = {pronoun: text.split().count(pronoun) for pronoun in sps_list}
        word_counts_spp = {pronoun: text.split().count(pronoun) for pronoun in spp_list}
        word_counts_tps = {pronoun: text.split().count(pronoun) for pronoun in tps_list}
        word_counts_tpp = {pronoun: text.split().count(pronoun) for pronoun in tpp_list}

        fps_counts = [(pronoun, count) for pronoun, count in word_counts_fps.items()]
        fpp_counts = [(pronoun, count) for pronoun, count in word_counts_fpp.items()]
        sps_counts = [(pronoun, count) for pronoun, count in word_counts_sps.items()]
        spp_counts = [(pronoun, count) for pronoun, count in word_counts_spp.items()]
        tps_counts = [(pronoun, count) for pronoun, count in word_counts_tps.items()]
        tpp_counts = [(pronoun, count) for pronoun, count in word_counts_tpp.items()]

        FPS = sum(word_counts_fps.values())
        FPP = sum(word_counts_fpp.values())
        SPS = sum(word_counts_sps.values())
        SPP = sum(word_counts_spp.values())
        TPS = sum(word_counts_tps.values())
        TPP = sum(word_counts_tpp.values())

    return doc_id, language, fps_counts, FPS, fpp_counts, FPP, sps_counts, SPS, spp_counts, SPP, tps_counts, TPS, tpp_counts, TPP

def apply_pronoun_analysis(df):
    if 'fps_counts' in df.columns and 'FPS' in df.columns and 'FPS_verbs' in df.columns and \
       'fpp_counts' in df.columns and 'FPP' in df.columns and 'FPP_verbs' in df.columns and \
       'sps_counts' in df.columns and 'SPS' in df.columns and 'SPS_verbs' in df.columns and \
       'spp_counts' in df.columns and 'SPP' in df.columns and 'SPP_verbs' in df.columns and \
       'tps_counts' in df.columns and 'TPS' in df.columns and 'TPS_verbs' in df.columns and \
       'tpp_counts' in df.columns and 'TPP' in df.columns and 'TPP_verbs' in df.columns:
        print("Columns already in DataFrame.")
        return df
    results = []
    for idx, row in df.iterrows():
        doc_id = idx
        text = row['clean_transcript']
        language = 'turkish'

        fpp_result = Pronoun_analysis(doc_id, text, language)

        pos_tags = row['pos_tag']
        FPS_verbs = [word for word, tag in pos_tags if tag == 'VERB' and word.endswith('m')]
        FPP_verbs = [word for word, tag in pos_tags if tag == 'VERB' and any(word.endswith(suffix) for suffix in ['ik', 'ık', 'ız', 'iz', 'uz', 'üz'])]
        SPS_verbs = [word for word, tag in pos_tags if tag == 'VERB' and any(word.endswith(suffix) for suffix in ['ın','in', 'un', 'ün'])]
        SPP_verbs = [word for word, tag in pos_tags if tag == 'VERB' and any(word.endswith(suffix) for suffix in ['niz','nız', 'nuz','nüz'])]
        TPS_verbs = [word for word, tag in pos_tags if tag == 'VERB' and any(word.endswith(suffix) for suffix in ['ı','i' ])]
        TPP_verbs = [word for word, tag in pos_tags if tag == 'VERB' and any(word.endswith(suffix) for suffix in ['lar', 'ler','lardı', 'lerdi', 'larmış','lermiş'])]

        results.append((*fpp_result, FPS_verbs, FPP_verbs, SPS_verbs, SPP_verbs, TPS_verbs, TPP_verbs))

    df_results = pd.DataFrame(results, columns=['doc_id', 'language', 'fps_counts', 'FPS', 'fpp_counts', 'FPP', 'sps_counts', 'SPS', 'spp_counts','SPP','tps_counts', 'TPS', 'tpp_counts', 'TPP',
                                                'FPS_verbs', 'FPP_verbs', 'SPS_verbs', 'SPP_verbs', 'TPS_verbs', 'TPP_verbs'])

    final_df = pd.concat([df, df_results[['fps_counts', 'FPS', 'FPS_verbs', 'fpp_counts', 'FPP', 'FPP_verbs', 'sps_counts', 'SPS', 'SPS_verbs', 'spp_counts', 'SPP', 'SPP_verbs',
                                          'tps_counts', 'TPS', 'TPS_verbs', 'tpp_counts', 'TPP', 'TPP_verbs']]], axis=1)

    return final_df

df = apply_pronoun_analysis(df)

pd.set_option('display.max_colwidth', 50)
pronoun_df = df[['clean_transcript', 'fps_counts', 'FPS', 'FPS_verbs','fpp_counts', 'FPP', 'FPP_verbs', 'sps_counts', 'SPS', 'SPS_verbs', 'spp_counts', 'SPP', 'SPP_verbs',
                 'tps_counts', 'TPS', 'TPS_verbs', 'tpp_counts', 'TPP', 'TPP_verbs']]

display(pronoun_df)


,clean_transcript,fps_counts,FPS,FPS_verbs,fpp_counts,FPP,FPP_verbs,sps_counts,SPS,SPS_verbs,spp_counts,SPP,SPP_verbs,tps_counts,TPS,TPS_verbs,tpp_counts,TPP,TPP_verbs
0,mi Evet günaydınlar herkese Merhabalar başlıkt...,"[(Ben, 22), (Beni, 0), (Bana, 1), (Benim, 2), ...",26.0,"[paylaşalım, bahsedeyim, geçiriyorum, alıyorum...","[(Biz, 0), (Bizi, 1), (Bize, 0), (Bizim, 1), (...",2.0,"[olabiliriz, ayrılabiliriz, kovabiliriz, baktı...","[(Sen, 0), (Seni, 0), (Sana, 0), (Senin, 0), (...",0.0,"[gelsin, Bakın, yıktın, bakmanın, kaybetmenin,...","[(Siz, 2), (Sizi, 0), (Size, 0), (Sizin, 0), (...",2.0,"[desteğiniz, yaşadığınız, yaşadığınız, etmeyec...","[(O, 4), (Onu, 1), (Ona, 0), (Onun, 1), (Onda,...",16.0,"[sevdi, yaşadıkları, vardı, gitti, sıkıldığı, ...","[(Onlar, 0), (Onları, 1), (Onlara, 0), (Onları...",1.0,"[yaşıyorlar, yürütüyorlar, diyorlar, ettiler, ..."
1,Evet arkadaşlar selam realistik kanalıma Hoşge...,"[(Ben, 23), (Beni, 0), (Bana, 0), (Benim, 8), ...",31.0,"[yükledim, bilmiyorum, yıkayacağım, bilmiyorum...","[(Biz, 3), (Bizi, 0), (Bize, 0), (Bizim, 0), (...",3.0,"[edersiniz, yeneriz, Affedersiniz, yaşıyoruz, ...","[(Sen, 1), (Seni, 0), (Sana, 0), (Senin, 0), (...",1.0,"[bakmayın, olsun, olsun, olsun, izleyin, gelin...","[(Siz, 1), (Sizi, 0), (Size, 0), (Sizin, 0), (...",1.0,"[edersiniz, Affedersiniz, geçiriyormuşsunuz, a...","[(O, 6), (Onu, 1), (Ona, 0), (Onun, 0), (Onda,...",27.0,"[ağırlaştığı, yapmayı, yapmayı, hissettiğimi, ...","[(Onlar, 0), (Onları, 0), (Onlara, 0), (Onları...",0.0,"[olmayanlar, izlemezler, sağlar, yiyorlar, gel..."
2,Evet arkadaşlar Merhabalar günaydınlar depresy...,"[(Ben, 21), (Beni, 1), (Bana, 0), (Benim, 9), ...",31.0,"[bahsettim, istiyorum, bileyim, yapayım, yapay...","[(Biz, 0), (Bizi, 0), (Bize, 0), (Bizim, 1), (...",1.0,"[biliyorsunuz, giriyoruz, alıyoruz, alıyoruz, ...","[(Sen, 4), (Seni, 0), (Sana, 0), (Senin, 0), (...",4.0,"[bakın, yapıyorsun, kendin, üzülüyorsun, üzülü...","[(Siz, 3), (Sizi, 0), (Size, 0), (Sizin, 0), (...",3.0,"[biliyorsunuz, bilmiyorsanız, biliyorsunuz, bi...","[(O, 8), (Onu, 0), (Ona, 0), (Onun, 0), (Onda,...",27.0,"[geçti, çıkmasını, giyinmesi, görüşmemesi, olm...","[(Onlar, 0), (Onları, 0), (Onlara, 1), (Onları...",1.0,"[dışladılar, etmediler, ettiler, kalktılar, ba..."
3,Herkese selamlar ayrılıkla mücadele edememe de...,"[(Ben, 18), (Beni, 0), (Bana, 1), (Benim, 4), ...",25.0,"[edelim, ediyorum, yapıyorum, aldığım, Geziyor...","[(Biz, 2), (Bizi, 0), (Bize, 0), (Bizim, 0), (...",2.0,"[geldiniz, biliyorsunuz, bahsediyoruz, alıyors...","[(Sen, 3), (Seni, 0), (Sana, 0), (Senin, 2), (...",5.0,"[çalışmayın, düşün, bakın, taşımayabilirsin, t...","[(Siz, 2), (Sizi, 0), (Size, 1), (Sizin, 0), (...",3.0,"[geldiniz, biliyorsunuz, alıyorsunuz, biliyors...","[(O, 7), (Onu, 0), (Ona, 1), (Onun, 2), (Onda,...",16.0,"[Açıldı, kapandı, eklendi, çalışmaları, yapmay...","[(Onlar, 0), (Onları, 0), (Onlara, 0), (Onları...",0.0,"[vermişler, yönelmeler, koymuyorlar, bahsediyo..."
4,Evet Merhabalar hoş geldik Hoş bulduk Nasılsın...,"[(Ben, 16), (Beni, 0), (Bana, 0), (Benim, 6), ...",23.0,"[kaldım, geçeceğim, çekmiştim, oldum, alamıyor...","[(Biz, 0), (Bizi, 0), (Bize, 0), (Bizim, 0), (...",0.0,"[geldik, görüşememiştik, yaptık, gördüyseniz, ...","[(Sen, 4), (Seni, 0), (Sana, 0), (Senin, 1), (...",6.0,"[bakın, Bakın, gelmesin, olun, alın, söylemeyi...","[(Siz, 1), (Sizi, 0), (Size, 0), (Sizin, 0), (...",1.0,"[gördüyseniz, olmayacaksınız, gidersiniz, atla...","[(O, 8), (Onu, 0), (Ona, 0), (Onun, 1), (Onda,...",33.0,"[çıktı, kaldı, olmasını, çıkmadı, geçti, patla...","[(Onlar, 0), (Onları, 1), (Onlara, 0), (Onları...",1.0,"[yakmışlar, alırlar, yaparlar, çekerler, zorun..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39,NaN,"[(Ben, 12), (Beni, 0), (Bana, 0), (Benim, 1), ...",13.0,"[arıyordum, arıyordum, başvurdum, dedim, başvu...","[(Biz, 0), (Bizi, 0), (Bize, 0), (Bizim, 0), (...",0.0,"[tredladık, biliyoruz, izliyoruz, izliyoruz, o...","[(Sen, 2), (Seni,

**Verbs:** These are the conjugated forms of the verb, carrying tense, mood,

*   **Verbs:** These are the conjugated forms of the verb, carrying tense, mood, aspect, person, and other grammatical markers.
For example:

    * paylaşalım ("let's share")
    * olabiliriz ("we might be")
    * bahsedeyim ("let me mention")

*   **Lemmas:** These are the base forms of the verbs, which correspond to their dictionary entries. They represent the neutral form, such as:

    * paylaş ("to share")
    * ol ("to be")
    * bahset ("to mention")

* Suffixes: These are the parts of the conjugated verb that deviate from the lemma. They include markers for tense, mood, negation, etc. For instance:

  * alım from paylaşalım: Includes the 1st-person plural mood marker.
  * abiliriz from olabiliriz: Indicates possibility and 1st-person plural.

In [9]:
import stanza
import pandas as pd

# Define vowels and consonants in turkish
VOWELS = {'a', 'e', 'ı', 'i', 'o', 'ö', 'u', 'ü'}
CONSONANTS = {'b', 'c', 'ç', 'd', 'f', 'g', 'ğ', 'h', 'j', 'k', 'l', 'm', 'n', 'p', 'r', 'ş', 't', 'v', 'y', 'z'}

# Suffixes to create passive voice
PASSIVE_SUFFIX_CONSONANT = {'al', 'el', 'ıl', 'il', 'ol', 'öl', 'ul', 'ül', 'an', 'en', 'ın', 'in', 'on', 'ön', 'un', 'ün'}
PASSIVE_SUFFIX_VOWEL = {'n', 'l'}

# Ensure to manage non-iterable values in pos_tag column
def extract_verbs(tags):
    if isinstance(tags, list):
        return [word for word, pos in tags if pos == "VERB"]
    return []

# Extract verbos from "pos_tag" column
df_verbs = pd.DataFrame({
    "verbs": df["pos_tag"].apply(extract_verbs)
})

# Initiate turkish pipeline
nlp = stanza.Pipeline('tr')
results = []

# Process verbs to get lemas
for _, row in df_verbs.iterrows():
    verbs = row['verbs']
    if isinstance(verbs, list):
        text = " ".join(verbs)
        doc = nlp(text)
        lemmas = [word.lemma for sentence in doc.sentences for word in sentence.words]
        results.append(lemmas)
    else:
        results.append([])

# Add lemmas to df
df_verbs['lemmas'] = results

# Function to extract the suffix of a verb comparing it to the lemma
def extract_suffix(verb, lemma):
    i = 0
    while i < len(verb) and i < len(lemma) and verb[i] == lemma[i]:
        i += 1
    return verb[i:]

# Apply function
df_verbs["suffixes"] = df_verbs.apply(
    lambda row: [
        extract_suffix(verb, lemma)
        for verb, lemma in zip(row["verbs"], row["lemmas"])
    ],
    axis=1
)

# Get verbs in passive voice
def get_passive_verbs(lemmas, verbs):
    passive_verbs = []

    for lemma, verb in zip(lemmas, verbs):
        if lemma and verb.startswith(lemma):  # Verify that verb comes from root/lemma
            suffix = verb[len(lemma):]  # Extract suffix

            # Roots that end in a consonat
            if lemma[-1] in CONSONANTS and any(suffix.startswith(ps) for ps in PASSIVE_SUFFIX_CONSONANT):
                passive_verbs.append(verb)

            # Roots that end in a vowel
            elif lemma[-1] in VOWELS and any(suffix.startswith(ps) for ps in PASSIVE_SUFFIX_VOWEL):
                passive_verbs.append(verb)

    return passive_verbs

# Apply get passive verbs function
df_verbs['passive_voice'] = df_verbs.apply(
    lambda row: get_passive_verbs(row['lemmas'], row['verbs']),
    axis=1
)

display(df_verbs.head())


INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/stanza_resources/resources.json
INFO:stanza:Loading these models for language: tr (Turkish):
| Processor | Package       |
-----------------------------
| tokenize  | imst          |
| mwt       | imst          |
| pos       | imst_charlm   |
| lemma     | imst_nocharlm |
| depparse  | imst_charlm   |
| ner       | starlang      |

INFO:stanza:Using device: cuda
INFO:stanza:Loading: tokenize
/usr/local/lib/python3.10/dist-packages/stanza/models/tokenization/trainer.py:82: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during

,verbs,lemmas,suffixes,passive_voice
0,"[paylaşalım, olabiliriz, bahsedeyim, geçiriyor...","[paylaş, ol, bahset, geçir, al, gel, gel, ayrı...","[alım, abiliriz, deyim, iyorum, ıyorum, irse, ...","[paylaşalım, açılmıştır, dinlenme, taşındım, b..."
1,"[oldu, edersiniz, yükledim, bilmiyorum, yıkaya...","[ol, et, yükle, bil, yıka, bil, çık, yap, sal,...","[du, dersiniz, dim, miyorum, yacağım, miyorum,...","[yıkılıp, yeneriz, olan, koşan, gelin, konuşal..."
2,"[biliyorsunuz, bahsettim, etmek, istiyorum, gi...","[bil, bahset, et, iste, gir, geç, ol, et, ol, ...","[iyorsunuz, tim, mek, iyorum, iyoruz, ti, acak...","[olan, süslenir, alan, süren, olan, bakın, yık..."
3,"[edememe, edememe, geldiniz, biliyorsunuz, bah...","[et, et, gel, bil, bahset, et, yap, et, kes, y...","[dememe, dememe, diniz, iyorsunuz, diyoruz, de...","[kapandı, eklendi, bakın, toparlanıyor, gelen,..."
4,"[geldik, bulduk, görüşememiştik, vermek, kaldı...","[gel, bul, görüş, ver, kal, geç, çek, ol, sür,...","[dik, duk, ememiştik, mek, dım, eceğim, miştim...","[kaynaklanıyor, küçülüyor, bakın, kırılır, açı..."
